In [87]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

In [ ]:
class Layer:
    def forward(self, X):
        raise NotImplementedError

    def backward(self, dY):
        raise NotImplementedError

    def update(self, lr):
        pass

class Dense(Layer):
    def __init__(self, in_features, out_features):
        self.W = np.random.randn(in_features, out_features) * 0.01
        self.b = np.zeros((1, out_features))

    def forward(self, X):
        self.X = X
        return X @ self.W + self.b

    def backward(self, dZ):
        m = self.X.shape[0]
        self.dW = self.X.T @ dZ
        self.db = np.sum(dZ, axis=0, keepdims=True)
        return dZ @ self.W.T

    def update(self, lr):
        self.W -= lr * self.dW
        self.b -= lr * self.db
      
class ReLU(Layer):
    def forward(self, Z):
        self.Z = Z
        return np.maximum(0, Z)

    def backward(self, dA):
        return dA * (self.Z > 0)
    
class MSELoss:
    def forward(self, Y_hat, Y):
        self.Y_hat = Y_hat
        self.Y = Y
        return np.mean((Y_hat - Y) ** 2)

    def backward(self):
        m = self.Y.shape[0]
        dy_hat =  2 * (self.Y_hat - self.Y) / m  
        #dy_hat = (dy_hat - dy_hat.mean(axis=0)) / dy_hat.std(axis=0)  
        return dy_hat

class LogisticLoss:
    def forward(self, Y_hat, Y):
        self.Y_hat = Y_hat
        self.Y = Y
        loss  = -np.mean(np.sum(Y * np.log(Y_hat ), axis=1, keepdims=True)) # - sum Y_i * Y_hat_i
        return loss
    def backward(self):
        return (self.Y_hat - self.Y) / self.Y.shape[0]

# softmax plus cross entropy are useful with multi-calss classification and more stable than sigmoid + MSE
# the output of softmax is propbabily which its range [0-1] and summation of all output layer is 1(y_hat)
# compute loss is very accurate
# TODO: Need to revise the Math later  
class SoftmaxCrossEntropy:
    def forward(self, Z, Y):
        self.Y = Y

        exp = np.exp(Z - np.max(Z, axis=1, keepdims=True))
        self.Y_hat = exp / np.sum(exp, axis=1, keepdims=True)

        loss = -np.mean(np.sum(Y * np.log(self.Y_hat + 1e-9), axis=1))
        return loss

    def backward(self):
        return self.Y_hat - self.Y


class sigmoid(Layer):
    def forward(self, z):
        self.a = 1 / (1 + np.exp(-z)) 
        return self.a
    
    def backward(self, da):
        return self.a * (1 - self.a) * da 

class softMax(Layer):
    def forward(self, Z):
        Z_shifted = Z - np.max(Z, axis=1, keepdims=True)  # stability
        exp = np.exp(Z_shifted)
        self.Y_hat = exp / np.sum(exp, axis=1, keepdims=True)
        return self.Y_hat
    
    def backward(self, da):
        return da   
 
class Model:
    def __init__(self):
        self.layers = []

    def add(self, layer):
        self.layers.append(layer)

    def forward(self, X):
        i = 0
        for layer in self.layers:
            X = layer.forward(X)
        return X

    def backward(self, grad):
        for layer in reversed(self.layers):
            grad = layer.backward(grad)

    def update(self, lr):
        for layer in self.layers:
            layer.update(lr)

In [89]:
# Load dataset
X, y = load_iris(return_X_y=True)

# Normalize features
X = (X - X.mean(axis=0)) / X.std(axis=0)

# One-hot encode labels
Y = np.eye(3)[y]

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, shuffle=True, random_state=1)


In [90]:
np.random.seed(0)

model = Model()
model.add(Dense(4, 12))
model.add(sigmoid())
model.add(Dense(12, 12))
model.add(sigmoid())
model.add(Dense(12, 3))  # Linear output
model.add(softMax())

loss_fn = LogisticLoss()


In [91]:
lr = 0.099
epochs = 20000

flag = True

for epoch in range(epochs):
    # Forward
    outputs = model.forward(X)
    if flag:
        print(outputs.shape)
        print(outputs[0])
        flag = False

    loss = loss_fn.forward(outputs, Y)

    # Backward
    grad = loss_fn.backward()
    model.backward(grad)

    # Update
    model.update(lr)

    if epoch % 300 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.6f}")


(150, 3)
[0.33367697 0.33118635 0.33513668]
Epoch 0, Loss: 1.098625
Epoch 300, Loss: 1.098612
Epoch 600, Loss: 1.098612
Epoch 900, Loss: 1.098611
Epoch 1200, Loss: 1.098611
Epoch 1500, Loss: 1.098610
Epoch 1800, Loss: 1.098610
Epoch 2100, Loss: 1.098609
Epoch 2400, Loss: 1.098608
Epoch 2700, Loss: 1.098607
Epoch 3000, Loss: 1.098605
Epoch 3300, Loss: 1.098603
Epoch 3600, Loss: 1.098600
Epoch 3900, Loss: 1.098595
Epoch 4200, Loss: 1.098586
Epoch 4500, Loss: 1.098572
Epoch 4800, Loss: 1.098547
Epoch 5100, Loss: 1.098492
Epoch 5400, Loss: 1.098357
Epoch 5700, Loss: 1.097916
Epoch 6000, Loss: 1.095486
Epoch 6300, Loss: 1.046460
Epoch 6600, Loss: 0.466413
Epoch 6900, Loss: 0.304451
Epoch 7200, Loss: 0.202210
Epoch 7500, Loss: 0.144805
Epoch 7800, Loss: 0.114659
Epoch 8100, Loss: 0.096600
Epoch 8400, Loss: 0.084517
Epoch 8700, Loss: 0.075920
Epoch 9000, Loss: 0.069569
Epoch 9300, Loss: 0.064733
Epoch 9600, Loss: 0.060951
Epoch 9900, Loss: 0.057923
Epoch 10200, Loss: 0.055452
Epoch 10500, Los

In [92]:
out = model.forward(X)
out = np.round(out)
accuracy = np.mean(out == Y)
print("Accuracy: ", accuracy)


Accuracy:  0.9911111111111112
